In [1]:
from Helpers import *

In [2]:
df_cancelled = pd.read_csv("../Data and descriptions/Case Rigshospitalet - Cancelled operations.csv", sep=';', low_memory=False)
df_complete = pd.read_csv("../Data and descriptions/Case Rigshospitalet - Completed operations.csv", sep = ';', low_memory=False)
datetime_variables = ['Dato', 'Pt ankommet til hospitalet', 'Planlagt stue klargøring start', 'Planlagt stue klargøring start', 'Stue klargøring start', 'Stue klargjort', 'Patient på stuen', 'Patient på stuen (Planlagt)', 'Anæstesistart', 'Anæstesi melder klar', 'Procedure start', 'Procedure slut', 'Patient klar til afgang', 'Patient forlader stuen (Planlagt)', 'Patient forlader stuen', 'Stue rengjort (Planlagt)', 'Stue rengøring start', 'Stue rengjort', 'I opvågning', 'Anæstesistop', 'Klar til udskrivelse efter opvågning', 'Patient forlader afdeling']

In [3]:
# remove outliers in the night (00:00-07:59)
procedure_start_tmp = pd.to_datetime(
    df_complete["Procedure start"],
    format="%Y-%m-%d %H:%M:%S,%f",
    errors="coerce"
)

before_rows = len(df_complete)
keep_mask = procedure_start_tmp.dt.hour.isna() | (procedure_start_tmp.dt.hour >= 8)
df_complete = df_complete.loc[keep_mask].copy()
after_rows = len(df_complete)

print(f"Removed {before_rows - after_rows} night operations (00:00-07:59)")

Removed 1891 night operations (00:00-07:59)


In [4]:
# Make datetime objects
for i in datetime_variables:
    df_complete[i] = pd.to_datetime(df_complete[i], format='%Y-%m-%d %H:%M:%S,%f')

In [5]:
df_complete["Individuel forsinkelse"] = df_complete["Overskredet (minutter)"] - df_complete["Forsinkelse (minutter)"]

In [6]:
# remove columns with staff and resources
df_complete_wo = df_complete.drop(columns=[col for col in df_complete.columns if col.startswith("Ressource")])
df_complete_wo = df_complete_wo.drop(columns=[col for col in df_complete_wo.columns if col.startswith("Staff")])

In [7]:
df_complete_NotAkut = get_dataframe_by_val_in_key(df_complete_wo,"Akut case (J/N)", "Nej")

In [ ]:
tid_df = df_complete_wo[["Speciale", "Procedure start", "Individuel forsinkelse"]].dropna().copy()
tid_df["Start-time"] = tid_df["Procedure start"].dt.hour

specialer = ["Alle"] + sorted(tid_df["Speciale"].dropna().unique().tolist())

def plot_forsinkelse_for_speciale(speciale):
    if speciale == "Alle":
        plot_df = tid_df
    else:
        plot_df = tid_df[tid_df["Speciale"] == speciale]

    stats = (
        plot_df.groupby("Start-time")["Individuel forsinkelse"]
        .agg(
            mean="mean",
            q1=lambda s: s.quantile(0.25),
            q3=lambda s: s.quantile(0.75)
        )
        .sort_index()
    )

    if stats.empty:
        print(f"Ingen data for Speciale = {speciale}")
        return

    timer = stats.index.to_numpy()
    mean_vals = stats["mean"].to_numpy()
    q1_vals = stats["q1"].to_numpy()
    q3_vals = stats["q3"].to_numpy()

    plot_to_index = 16

    plt.figure(figsize=(14, 5))
    plt.fill_between(timer[:plot_to_index], q1_vals[:plot_to_index], q3_vals[:plot_to_index], alpha=0.25, label="1. og 3. kvartil")
    plt.plot(timer[:plot_to_index], mean_vals[:plot_to_index], marker="o", linewidth=2, label="Gennemsnit")
    plt.title(f"Individuel forsinkelse som funktion af tidspunkt på dagen - {speciale}")
    plt.xlabel("Procedure start (time of day)")
    plt.ylabel("Individuel forsinkelse (minutter)")
    plt.xticks(timer[:plot_to_index], [f"{int(t):02d}:00" for t in timer[:plot_to_index]], rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


dropdown = widgets.Dropdown(
    options=specialer,
    value="Alle",
    description="Speciale:"
)

output = widgets.interactive_output(
    plot_forsinkelse_for_speciale,
    {"speciale": dropdown}
)

display(dropdown, output)

Dropdown(description='Speciale:', options=('Alle', 'Alloplastik', 'Anæstesiologi', 'Brystkirurgi', 'Børnekirur…

Output()

In [9]:
def plot_planned_operations_throughout_the_day_for_speciale(speciale):
    if speciale == "Alle":
        df_in_question = df_complete_NotAkut
    else:
        df_in_question = df_complete_NotAkut[df_complete_NotAkut["Speciale"] == speciale].copy()

    df_in_question = df_in_question.dropna(subset=["Procedure start"])
    df_in_question["Start-time"] = df_in_question["Procedure start"].dt.hour

    counts = (
        df_in_question["Start-time"]
        .value_counts()
        .reindex(range(24), fill_value=0)
        .sort_index()
    )

    counts_df = pd.DataFrame({
        "Time": [f"{h:02d}:00" for h in counts.index],
        "Count": counts.values,
    })
    counts_df["Share (%)"] = (counts_df["Count"] / counts_df["Count"].sum() * 100).round(2)

    plt.figure(figsize=(14, 4))
    plt.bar(counts_df["Time"], counts_df["Count"])
    plt.title("Øjenkirurgi operations by time of day")
    plt.xlabel("Procedure start (hour)")
    plt.ylabel("Number of operations")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    return None

dropdown = widgets.Dropdown(
    options=specialer,
    value="Alle",
    description="Speciale:"
)

output = widgets.interactive_output(
    plot_planned_operations_throughout_the_day_for_speciale,
    {"speciale": dropdown}
)

display(dropdown, output)

Dropdown(description='Speciale:', options=('Alle', 'Alloplastik', 'Anæstesiologi', 'Brystkirurgi', 'Børnekirur…

Output()

In [ ]:
target_gang = 107621
target_dato = pd.Timestamp("2024-03-07")

df_of_Operationsgang_and_Dato(df_complete_wo, target_gang=target_gang, target_dato=target_dato)

,Case-ID Anonymous,Patient Alder,Speciale,Stue,Operationsgang ID,Akut case (J/N),Dato,Pt ankommet til hospitalet,Planlagt stue klargøring start,Stue klargøring start,...,Anæstesistop,Klar til udskrivelse efter opvågning,Patient forlader afdeling,Forsinkelse (minutter),Overskredet (minutter),Forsinkelsesårsag,Procedure - Tekst & ID,Aktionsdiagnose - Kode & tekst,Aktionsdiagnose - Gruppe,Individuel forsinkelse
0,28,29,Øre-næse-hals,OPNORD 23.240 (H),107621,Nej,2024-03-07,2024-03-07 10:51:00,2024-03-07 12:05:00,2024-03-07 13:05:00,...,NaT,NaT,2024-03-07 15:24:00,95.0,36.0,NaN,AURES ALATAE [1070010290],DQ175: Aures alatae,"Medfødte misdannelser i øje, øre, ansigtet og ...",-59.0
724,15552,96,Plastikkirurgi,OPNORD 13.111 (H),107621,Nej,2024-03-07,2024-03-06 18:33:00,2024-03-07 12:30:00,NaT,...,NaT,NaT,2024-03-07 15:44:00,96.0,66.0,NaN,"EXC. CANCER UE, LA + DELHUD [1070015267]",DC447: Anden hudkræft på underekstremitet,Kræftsygdomme,-30.0
796,17225,67,Plastikkirurgi,OPNORD 13.115 (H),107621,Ja,2024-03-07,2024-03-07 14:02:00,2024-03-07 14:00:00,2024-03-07 14:07:00,...,NaT,NaT,2024-03-07 14:32:00,7.0,-23.0,NaN,"EXC. CANCER ØVRIGE ANSIGT/HOVED/HALS, LA [1070...",DC443: Anden hudkræft i ansigtet med anden ell...,Kræftsygdomme,-30.0
803,17383,54,Plastikkirurgi,OPNORD 303,107621,Nej,2024-03-07,2024-03-07 10:36:00,2024-03-07 11:00:00,2024-03-07 11:06:00,...,NaT,NaT,2024-03-07 12:00:00,10.0,0.0,NaN,EXCISION AF MALIGNT MELANOM - KRÆFTPAKKE [1070...,DZ031P: Observation pga. mistanke om kræft i hud,Personer i kontakt med sundhedsvæsenet med hen...,-10.0
840,18200,56,Plastikkirurgi,OPNORD 13.115 (H),107621,Ja,2024-03-07,2024-03-07 11:57:00,2024-03-07 12:30:00,2024-03-07 12:11:00,...,NaT,NaT,2024-03-07 12:55:00,-12.0,-45.0,NaN,"EXC. CANCER ØVRIGE ANSIGT/HOVED/HALS, LA [1070...",DC443: Anden hudkræft i ansigtet med anden ell...,Kræftsygdomme,-33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99120,17309,74,Plastikkirurgi,OPNORD 303,107621,Nej,2024-03-07,2024-03-07 08:55:00,2024-03-07 09:00:00,2024-03-07 09:01:00,...,NaT,NaT,2024-03-07 09:58:00,1.0,-2.0,NaN,EXCISION AF MALIGNT MELANOM - KRÆFTPAKKE [1070...,DZ031P: Observation pga. mistanke om kræft i hud,Personer i kontakt med sundhedsvæsenet med hen...,-3.0
99121,17332,57,Plastikkirurgi,OPNORD 303,107621,Nej,2024-03-07,2024-03-07 09:35:00,2024-03-07 10:00:00,2024-03-07 10:01:00,...,NaT,NaT,2024-03-07 11:05:00,6.0,5.0,NaN,EXCISION AF MALIGNT MELANOM - KRÆFTPAKKE [1070...,DZ031P: Observation pga. mistanke om kræft i hud,Personer i kontakt med sundhedsvæsenet med hen...,-1.0
99124,17460,49,Plastikkirurgi,OPNORD 303,107621,Nej,2024-03-07,2024-03-07 12:22:00,2024-03-07 12:30:00,2024-03-07 12:31:00,...,NaT,NaT,2024-03-07 13:24:00,6.0,-6.0,NaN,EXCISION AF MALIGNT MELANOM - KRÆFTPAKKE [1070...,DZ031P: Observation pga. mistanke om kræft i hud,Personer i kontakt med sundhedsvæsenet med hen...,-12.0
99125,17467,36,Plastikkirurgi,OPNORD 303,107621,Nej,2024-03-07,2024-03-07 13:34:00,2024-03-07 13:30:00,2024-03-07 13:36:00,...,NaT,NaT,2024-03-07 14:21:00,6.0,-9.0,NaN,EXCISION AF MALIGNT MELANOM - KRÆFTPAKKE [1070...,DZ031P: Observation pga. mistanke om kræft i hud,Personer i kontakt med sundhedsvæsenet med hen...,-15.0


In [ ]:
target_stue = "OPNORD 23.240 (H)"
target_dato = pd.Timestamp("2024-03-07")

df_room_date = df_of_Stue_and_Dato(df_complete_wo, target_stue=target_stue)


tid_df = df_room_date[["Speciale", "Procedure start", "Individuel forsinkelse"]].dropna().copy()
tid_df["Start-time"] = tid_df["Procedure start"].dt.hour

specialer = ["Alle"] + sorted(tid_df["Speciale"].dropna().unique().tolist())


dropdown = widgets.Dropdown(
    options=specialer,
    value="Alle",
    description="Speciale:"
)

output = widgets.interactive_output(
    plot_forsinkelse_for_speciale,
    {"speciale": dropdown}
)

display(dropdown, output)

dropdown = widgets.Dropdown(
    options=specialer,
    value="Alle",
    description="Speciale:"
)

output = widgets.interactive_output(
    plot_planned_operations_throughout_the_day_for_speciale,
    {"speciale": dropdown}
)

display(dropdown, output)

Dropdown(description='Speciale:', options=('Alle', 'Tand- mund- og kæbekirurgi', 'Øre-næse-hals'), value='Alle…

Output()